In [1]:
!pip install glfw PyOpenGL Pillow PyGLM

   ---------------------------------------- 0.0/559.5 kB ? eta -:--:--
   ------------------ --------------------- 262.1/559.5 kB ? eta -:--:--
   ---------------------------------------- 559.5/559.5 kB 4.9 MB/s  0:00:00
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ------------------- -------------------- 1.6/3.2 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 3.2/3.2 MB 9.3 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 9.3 MB/s  0:00:00

   ---------------------------------------- 0/3 [PyOpenGL]
   ---------------------------------------- 0/3 [PyOpenGL]
   ---------------------------------------- 0/3 [PyOpenGL]
   ---------------------------------------- 0/3 [PyOpenGL]
   ---------------------------------------- 0/3 [PyOpenGL]
   ---------------------------------------- 0/3 [PyOpenGL]
   ---------------------------------------- 0/3 [PyOpenGL]
   --


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\chobotam\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
import glfw
from OpenGL.GL import *
from OpenGL.GL.shaders import compileShader, compileProgram
import numpy as np
import glm
from PIL import Image
import ctypes

# Налаштування вікна
SCR_WIDTH, SCR_HEIGHT = 800, 600

# Глобальні змінні
active_cube = 0
positions = [glm.vec3(0,0,0), glm.vec3(2,0,0), glm.vec3(-2,0,0)]
angle = 0.0
camPos = glm.vec3(0.0, 0.0, 5.0)
camFront = glm.vec3(0.0, 0.0, -1.0)
camUp = glm.vec3(0.0, 1.0, 0.0)
yaw, pitch = -90.0, 0.0
lastX, lastY = 400, 300
firstMouse = True

def mouse_callback(window, xpos, ypos):
    global firstMouse, yaw, pitch, lastX, lastY, camFront
    if firstMouse: lastX, lastY = xpos, ypos; firstMouse = False
    xoffset, yoffset = xpos - lastX, lastY - ypos
    lastX, lastY = xpos, ypos
    sensitivity = 0.1
    yaw += xoffset * sensitivity; pitch += yoffset * sensitivity
    pitch = max(min(pitch, 89.0), -89.0)
    front = glm.vec3(np.cos(glm.radians(yaw)) * np.cos(glm.radians(pitch)), np.sin(glm.radians(pitch)), np.sin(glm.radians(yaw)) * np.cos(glm.radians(pitch)))
    camFront = glm.normalize(front)

def main():
    if not glfw.init(): return
    glfw.window_hint(glfw.STENCIL_BITS, 8)
    glfw.window_hint(glfw.DEPTH_BITS, 24)
    window = glfw.create_window(SCR_WIDTH, SCR_HEIGHT, "3D Cubes Lab", None, None)
    glfw.make_context_current(window)
    glfw.set_cursor_pos_callback(window, mouse_callback)
    glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)
    glEnable(GL_DEPTH_TEST)
    glEnable(GL_STENCIL_TEST) 

    # Шейдери
    vs = compileShader("""#version 330 core
    layout (location = 0) in vec3 aPos; layout (location = 1) in vec2 aTex;
    out vec2 TexCoord; uniform mat4 model; uniform mat4 view; uniform mat4 proj;
    void main() { gl_Position = proj * view * model * vec4(aPos, 1.0); TexCoord = aTex; }""", GL_VERTEX_SHADER)
    fs = compileShader("""#version 330 core
    out vec4 FragColor; in vec2 TexCoord;
    uniform sampler2D tex; uniform vec3 color;
    void main() { FragColor = texture(tex, TexCoord) * vec4(color, 1.0); }""", GL_FRAGMENT_SHADER)
    shader = compileProgram(vs, fs)

    # Вершини куба
    verts = np.array([-0.5,-0.5,-0.5,0,0, 0.5,-0.5,-0.5,1,0, 0.5,0.5,-0.5,1,1, 0.5,0.5,-0.5,1,1, -0.5,0.5,-0.5,0,1, -0.5,-0.5,-0.5,0,0,
                      -0.5,-0.5,0.5,0,0, 0.5,-0.5,0.5,1,0, 0.5,0.5,0.5,1,1, 0.5,0.5,0.5,1,1, -0.5,0.5,0.5,0,1, -0.5,-0.5,0.5,0,0,
                      -0.5,0.5,0.5,1,0, -0.5,0.5,-0.5,1,1, -0.5,-0.5,-0.5,0,1, -0.5,-0.5,-0.5,0,1, -0.5,-0.5,0.5,0,0, -0.5,0.5,0.5,1,0,
                       0.5,0.5,0.5,1,0, 0.5,0.5,-0.5,1,1, 0.5,-0.5,-0.5,0,1, 0.5,-0.5,-0.5,0,1, 0.5,-0.5,0.5,0,0, 0.5,0.5,0.5,1,0,
                      -0.5,-0.5,-0.5,0,1, 0.5,-0.5,-0.5,1,1, 0.5,-0.5,0.5,1,0, 0.5,-0.5,0.5,1,0, -0.5,-0.5,0.5,0,0, -0.5,-0.5,-0.5,0,1,
                      -0.5,0.5,-0.5,0,1, 0.5,0.5,-0.5,1,1, 0.5,0.5,0.5,1,0, 0.5,0.5,0.5,1,0, -0.5,0.5,0.5,0,0, -0.5,0.5,-0.5,0,1], dtype='f')

    vao = glGenVertexArrays(1); vbo = glGenBuffers(1)
    glBindVertexArray(vao); glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, verts.nbytes, verts, GL_STATIC_DRAW)
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, 20, None); glEnableVertexAttribArray(0)
    glVertexAttribPointer(1, 2, GL_FLOAT, GL_FALSE, 20, ctypes.c_void_p(12)); glEnableVertexAttribArray(1)

    tex = glGenTextures(1); glBindTexture(GL_TEXTURE_2D, tex)
    img = Image.open("tex1.jpg").transpose(Image.FLIP_TOP_BOTTOM)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img.width, img.height, 0, GL_RGB, GL_UNSIGNED_BYTE, img.convert("RGB").tobytes())
    glGenerateMipmap(GL_TEXTURE_2D)

    lastFrame = glfw.get_time()
    while not glfw.window_should_close(window):
        global angle, active_cube, camPos
        dt = glfw.get_time() - lastFrame; lastFrame = glfw.get_time(); angle += 50.0 * dt
        
        # Керування
        speed = 2.5 * dt
        if glfw.get_key(window, glfw.KEY_W) == glfw.PRESS: camPos += camFront * speed
        if glfw.get_key(window, glfw.KEY_S) == glfw.PRESS: camPos -= camFront * speed
        if glfw.get_key(window, glfw.KEY_1) == glfw.PRESS: active_cube = 0
        if glfw.get_key(window, glfw.KEY_2) == glfw.PRESS: active_cube = 1
        if glfw.get_key(window, glfw.KEY_3) == glfw.PRESS: active_cube = 2
        if glfw.get_key(window, glfw.KEY_UP) == glfw.PRESS: positions[active_cube].y += speed
        if glfw.get_key(window, glfw.KEY_DOWN) == glfw.PRESS: positions[active_cube].y -= speed
        if glfw.get_key(window, glfw.KEY_LEFT) == glfw.PRESS: positions[active_cube].x -= speed
        if glfw.get_key(window, glfw.KEY_RIGHT) == glfw.PRESS: positions[active_cube].x += speed

        glClearColor(0.1, 0.1, 0.1, 1.0)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT | GL_STENCIL_BUFFER_BIT)
        
        view, proj = glm.lookAt(camPos, camPos + camFront, camUp), glm.perspective(glm.radians(45.0), 800/600, 0.1, 100.0)
        glUseProgram(shader)
        glUniformMatrix4fv(glGetUniformLocation(shader, "view"), 1, GL_FALSE, glm.value_ptr(view))
        glUniformMatrix4fv(glGetUniformLocation(shader, "proj"), 1, GL_FALSE, glm.value_ptr(proj))

        for i, pos in enumerate(positions):
            model = glm.rotate(glm.translate(glm.mat4(1.0), pos), glm.radians(angle if i == active_cube else 0), glm.vec3(0,1,0))
            # Стенсил маска
            glStencilFunc(GL_ALWAYS, 1, 0xFF); glStencilOp(GL_KEEP, GL_KEEP, GL_REPLACE); glStencilMask(0xFF if i == active_cube else 0x00)
            glUniformMatrix4fv(glGetUniformLocation(shader, "model"), 1, GL_FALSE, glm.value_ptr(model)); glUniform3f(glGetUniformLocation(shader, "color"), 1, 1, 1)
            glDrawArrays(GL_TRIANGLES, 0, 36)
            # Рамка
            if i == active_cube:
                glStencilFunc(GL_NOTEQUAL, 1, 0xFF); glStencilMask(0x00); glDisable(GL_DEPTH_TEST)
                glUniformMatrix4fv(glGetUniformLocation(shader, "model"), 1, GL_FALSE, glm.value_ptr(glm.scale(model, glm.vec3(1.1))))
                glUniform3f(glGetUniformLocation(shader, "color"), 1, 0, 0); glDrawArrays(GL_TRIANGLES, 0, 36)
                glEnable(GL_DEPTH_TEST); glStencilMask(0xFF)
        
        glfw.swap_buffers(window); glfw.poll_events()
    glfw.terminate()

if __name__ == "__main__": main()